## **Resources**

> **Playlist**

[CampusX_Playlist](https://www.youtube.com/playlist?list=PLKnIA16_Rmva_oZ9F4ayUu9qcWgF7Fyc0)

> **Docs**

[MCP_Official_Docs](https://modelcontextprotocol.io/docs/getting-started/intro)

[MCP_Is_Dead](https://medium.com/@alonisser/mcp-is-dead-or-mcp-vs-skills-revisited-daaa51b9a519)

> **Best Open Source MCP Servers**

[MCP_MARKETPLACE](https://mcpmarket.com/leaderboards)

[Glama.ai](https://glama.ai/mcp/servers)

[Awesome-MCP-Servers](https://github.com/punkpeye/awesome-mcp-servers)

<hr>
<hr>

## **Intro to MCP(Model Context Protocol)**

MCP (Model Context Protocol) is an open-source standard for connecting AI applications to external systems.


Using `MCP`, AI applications like Claude or ChatGPT can connect to data sources (e.g. local files, databases), tools (e.g. search engines, calculators) and workflows (e.g. specialized prompts)—enabling them to access key information and perform tasks.

Think of MCP lik a USB-C port for AI applications. Just as USB-C provides a standardized way to connect electronic devices, `MCP` provides a standardized way for AI applications to connect to external systems. This allows AI applications to access a wider range of information and capabilities, making them more powerful and versatile.

To understand better refer to the image below: 

<img src="../notes_images/mcp/mcp.png" alt="MCP Overview" width="1000"/>

> MCP was developed by Anthropic, the company behind the Claude AI assistant, and is now an open standard that can be implemented by any AI application.

<hr>

## **What can MCP enable?**

* Agents can access your Google Calender and Notion, acting as a more personalized AI assistant.

* Claude Code can generate an entire web app using a Figma design.

* Enterprise chatbots can connect to multiple databases across an organization, empowering users to analyze data using chat.

* AI models can create 3D designs on Blender and print them out using a 3D printer.

## **MCP Ecosystem Support**

MCP is an open protocol supported across a wide range of clients and servers. AI assistants like `Claude` and `ChatGPT`, development tools like `Vs Code`, `Cursor`, `MCPJam` and others all support MCP - making it easy to build once and intergerate anywhere.

<hr>

## **MCP Archtecture Overview**

[MCP_Architecture](https://modelcontextprotocol.io/docs/learn/architecture)

MCP follows a client server architecture where an MCP host - an application or agentic harness like `Claude Code` establishes connections to one or more MCP servers. The `MCP` host accomplishes this by creating one `MCP` client for each `MCP` server. Each `MCP` client maintains a dedicated connection with its corresponding `MCP` server, enabling seamless communication and interaction between the host and the servers.

Local MCP servers that use the `STDIO` transport typically serve a single MCP client, whereas remote MCP servers that use the `Streaming HTTP` transport will serve many MCP clients.

### **Components of MCP Architecture:**

**MCP Host**: The MCP host is the central application or agentic harness that initiates and manages connections to MCP servers. It can be an AI assistant like `Claude Code` or a development tool like `Vs Code`. The host is responsible for coordinating and managing interactions with one or multiple MCP clients.

**MCP Client**: An MCP client maintains a connection to an MCP server and obtains context from an MCP server for the MCP host to use. Each MCP client is dedicated to a specific MCP server, allowing for efficient communication and data exchange between the host and the server.

**MCP Server**: An MCP server provides context and services to the MCP host through its corresponding MCP client. MCP servers can be local, using the `STDIO` transport, or remote, using the `Streaming HTTP` transport. Local MCP servers typically serve a single MCP client, while remote MCP servers can serve multiple MCP clients simultaneously.

### **Understand with an Example:**

Visual Studio Code (VS Code) acts as an MCP host. When Visual Studio Code establishes a connection to an MCP server, such as `Sentry MCP Server` (Sentry is a popular error tracking and monitoring platform), the Visual Studio Code runtime instantiates an MCP client object that maintains the connection to the `Sentry MCP Server`. When Visual Studio Code subsequently connects to another MCP server, such as the `Local File System Server`, the Visual Studio Code runtime instantiates an additional client object to maintain this second connection. In this example, Visual Studio Code is the MCP host, while the `Sentry MCP Server` and the `Local File System Server` are the MCP servers that provide context and services to Visual Studio Code through their respective MCP clients.

<img src="../notes_images/mcp/architecture.png" alt="MCP Architecture" width="1000"/>

<hr>

**JSON RPC 2.0**

RPC (Remote Procedure Call) is a protocol that allows a program to execute a procedure (subroutine) on another computer as if it were local, hiding the details of network communication and data transfer. This abstraction makes it easier to build distributed applications.

`Request`

```json

{
  "jsonrpc": "2.0",
  "method": "subtract",
  "params": [42, 23],
  "id": 1
}

```

`Response`

```json

{
  "jsonrpc": "2.0",
  "result": 19,
  "id": 1
}

```

<hr>

## **Layers in MCP Architecture**

MCP consists of two layers: 

### **Data Layer**

The data layer defines the `JSON-RPC` based protocol for client-server communication, including lifecycle management, and core primitives, such as tools, resources, prompts, and notifications. 

This layer includes: 

**Lifecycle Management**: Handles connection initialization, capability negotiation, and connection termination between clients and servers

**Server Features**: Enables servers to provide core functionality including tools for AI actions, resources for context data, and prompts for interaction templates from and to the client

**Client Features**: Enables servers to ask the client to sample from the host LLM, elicit input from the user, and log messages to the client

**Utility Features**: Supports additional capabilities like notifications for real-time updates and progress tracking for long-running operations

### **Transport Layer**

The transport layer manages communication channels and authentication between clients and servers. It handles connection establishment, message framing, and secure communication between MCP participants.

MCP supports two transport mechanisms: 

**STDIO Transport**: Uses standard input/output streams for direct process communication between local processes on the same machine, providing optimal performance with no network overhead.

- The `stdin` and `stdout` streams are used as transport layer between the client and server, allowing for efficient communication without the need for network protocols.

> The host (Vs Code) spawns a Local MCP Server as a `subprocess` (Parent - Child relationship). Since the `MCP Server` is a child process of the host, it has the complete control over the `STDIO` streams, with which the host writes `JSON-RPC` requests to the server's `stdin` and reads responses from the server's `stdout`. This setup allows for efficient communication between the host and the local MCP server without the need for network protocols.

> Due `STDIO` transport's the communication between the client and server becomes very fast because it eliminates the overhead associated with network communication, making it ideal for local interactions where low latency is crucial. If we had started the `MCP Server` as a separate program via some backend service, we would have had to use the `Streaming HTTP` transport, which would have introduced additional latency due to network communication.

**Streamable HTTP transport**: Uses `HTTP POST` for client-to-server messages with optional `Server-Sent Events` (SSE) for streaming capabilities. This transport enables remote server communication and supports standard HTTP authentication methods including bearer tokens, API keys, and custom headers. MCP recommends using `OAuth` to obtain authentication tokens.

- HTTP is used because it supports standard `Authentication` methods, like bearer tokens, API keys, and custom headers, which are essential for secure communication between clients and servers. Additionally, HTTP is a widely adopted protocol that can easily traverse firewalls and proxies, making it suitable for remote communication.

> SSE is used because instead of sending large JSON blob, the server can stream chunks of data to the client as they become available, allowing for real-time updates and reducing latency in scenarios where the server needs to send large amounts of data or provide progress updates for long-running operations.

> Note: There are two types of servers in `MCP`: Local servers that use the `STDIO` transport and remote servers that use the `Streaming HTTP` transport. Local servers typically serve a single client, while remote servers can serve multiple clients simultaneously.

<img src="../notes_images/mcp/archi2.png" alt="MCP Architecture Layers" width="1000"/>

<hr>


## **MCP Primitives**

MCP primitives are the most important concept within MCP. They define what clients and servers can offer each other. These primitives specify the types of contextual information that can be shared with AI applications and the range of actions that can be performed.

**Tools**: Executable functions that AI applications can invoke to perform actions (e.g., file operations, API calls, database queries)

- `tools/list`: Allows clients to discover available tools on the server

- `tools/call`: Enables clients to execute a specific tool with provided arguments

**Resources**: Data sources that provide contextual information to AI applications (e.g., file contents, database records, API responses)

- `resources/list`: Allows clients to discover available resources on the server

- `resources/get`: Enables clients to retrieve the content of a specific resource

- `resources/subscribe`: Allows clients to subscribe to updates for a specific resource, receiving notifications when the resource changes

**Prompts**: Reusable templates that help structure interactions with language models (e.g., system prompts, few-shot examples)

Each primitive type has associated methods for discovery `(*/list)`, retrieval `(*/get)`, and in some cases, execution (tools/call). MCP clients will use the `*/list` methods to discover available primitives. For example, a client can first list all available tools (tools/list) and then execute them. This design allows listings to be dynamic.

- `prompts/list`: Allows clients to discover available prompts on the server

- `prompts/get`: Enables clients to retrieve the content of a specific prompt

As a concrete example, consider an MCP server that provides context about a database. It can expose tools for querying the database, a resource that contains the schema of the database, and a prompt that includes few-shot examples for interacting with the tools.

### **JSON RPC Formats**

**Request**

```json

{
  "jsonrpc": "2.0",
  "method": "tools/list",
  "params": {
    "filter": {
      "type": "file"
    }
  },
  "id": 1
}

```

**Response**

```json

{
  "jsonrpc": "2.0",
  "result": {
    "tools": [
      {
        "id": "read_file",
        "name": "Read File",
        "description": "Reads the content of a file from the local file system.",
        "parameters": {
          "type": "object",
          "properties": {
            "file_path": {
              "type": "string",
              "description": "The path to the file to be read."
            }
          },
          "required": ["file_path"]
        }
      }
    ]
  },
  "id": 1
}

```

<hr>

## **Why JSON RPC for MCP Instead of REST or GraphQL?**

Remote Procedure Call (RPC) is lightweight compared to REST or GraphQL

JSON RPC supports bi-directional communication, allowing both clients and servers to initiate requests and send responses. This is particularly beneficial for real-time interactions and scenarios where low latency is crucial, such as in AI applications that require quick access to tools and resources.

JSON RPC supports notifications, which are messages sent from the server to the client without expecting a response. This allows servers to push updates or events to clients in real-time, enhancing the interactivity and responsiveness of AI applications.

Moreover, `JSON RPC` is transport agnostic, meaning it can be used over various communication channels (e.g., HTTP, WebSockets, StDIO) without being tied to a specific protocol. This flexibility allows MCP to support both local and remote servers seamlessly, catering to a wide range of use cases and deployment scenarios.

Also, `JSON RPC` supports batching of requests, enabling clients to send multiple requests in a single message and receive corresponding responses in a single batch. This can improve efficiency and reduce overhead in scenarios where multiple operations need to be performed in quick succession.

**How REST and GraphQL are not lightweight?**

- REST and GraphQL require more complex request and response structures, including HTTP headers, status codes, and often more verbose payloads. Due to this, creating HTTP requests and parsing responses can involve more overhead and complexity compared to the straightforward method invocation and result retrieval in RPC.

- RPC allows for more direct and efficient communication between clients and servers, as it abstracts away the complexities of HTTP and focuses on the core functionality of invoking methods and receiving results. This makes it particularly well-suited for real-time interactions and scenarios where low latency is crucial, such as in AI applications that require quick access to tools and resources.

<hr>

## **MCP Lifecycle**

There are three main stages in the lifecycle of an MCP connection:

### **Initialization**

[Clinet_Side_MCP](https://modelcontextprotocol.io/docs/learn/client-concepts)

The initialization phase `MUST` be the first interaction between client and server. 

- Estblish `Protocol` version compatibility

- Exchange and negotiate `Capabilities` (e.g. supported primitives, authentication methods)

At first, a client say `Visual Studio Code` will send an `initialize` request to the server (e.g. `Sentry MCP Server`) to establish a connection and negotiate capabilities: 

```json

{
  "jsonrpc": "2.0",
  "method": "initialize",
  "params": {
    "protocolVersion": "1.0",
    "clientCapabilities": {
      "roots" : {"listChanged": true},
      "sampling": {}
    }
  },
  "id": 1,
  "clientInfo": {
    "name": "Visual Studio Code",
    "version": "1.0.0"
  }
}

```

Then, the server will respond with its own capabilities and confirm the connection:

```json

{
  "jsonrpc": "2.0",
  "result": {
    "protocolVersion": "1.0",
    "serverCapabilities": {
      "tools": {
        "list": true,
        "call": true
  },
  "id": 1,
  "serverInfo": {
    "name": "Sentry MCP Server",
    "version": "1.0.0"
  }, 
  "instructions": "Welcome to Sentry MCP Server! You can use the tools provided to interact with Sentry's API. For example, you can list available tools using 'tools/list' and call specific tools using 'tools/call'. If you have any questions or need assistance, feel free to ask!"
}

```

**Core Clinet Features**

| Feature | Explanation | Example |
|---|---|---|
| `Elicitation` | Elicitation enables servers to request specific information from users during interactions, providing a structured way for servers to gather information on demand. | A travel booking server may ask for the user’s seat preference, room type, or contact number to finalize a booking. |
| `Roots` | Roots allow clients to specify which directories servers should focus on, communicating intended scope through a coordination mechanism. | A travel booking server may be given access to a specific directory from which it can read a user’s calendar. |
| `Sampling` | Sampling allows servers to request LLM completions through the client, enabling an agentic workflow. This approach puts the client in complete control of user permissions and security measures. | A travel booking server may send a list of flights to an LLM and request that the LLM pick the best flight for the user. |

**Core Server Features**

| Feature | Explanation | Examples | Who controls it |
|---|---|---|---|
| `Tools` | Callable actions exposed by the server that the model can choose to execute when needed. They can perform side effects like writes, API calls, or file updates. | Search flights<br>Send messages<br>Create calendar events | Model |
| `Resources` | Read-only context providers that expose structured or unstructured data to the client/model. | Retrieve documents<br>Access knowledge bases<br>Read calendars | Application |
| `Prompts` | Reusable prompt templates provided by the server to guide the model in specific workflows using available tools/resources. | Plan a vacation<br>Summarize meetings<br>Draft an email | User |


> During initialization, the client and server exchange information about their capabilities, such as tools, resources, and prompts they support. This allows both parties to understand what functionalities are available and how they can interact with each other effectively during the operation phase.

<hr>

### **Operation**

During the operation phase, the client and server exchange messages according to the negotiated capabilities.

- Respect the negotiated protocol version

- Only use capabilities that were successfully negotiated during initialization

**Capabilities Discovery**

During the initialization phase, the client and server exchange information about their capabilities and what functionalities they support. But during the operation phase, the client can also discover new capabilities that the server may have added after initialization. For example, a server might add new tools or resources dynamically, and the client can use the `*/list` methods to discover these new capabilities without needing to reinitialize the connection.

**Tool Calling**

LLM models can tell the `Host` to call tools on the `Server` by sending a `tools/call` request with the `name` of the tool and the required `parameters`. The server will execute the tool and return the result to the client, which can then be used by the LLM for further processing or decision-making.

```json
// Client Request to call a tool named "search_flights" with specific arguments
{
  "jsonrpc": "2.0",
  "method": "tools/call",
  "params": {
    "name": "search_flights",
    "arguments": {
      "origin": "New York",
      "destination": "San Francisco",
      "date": "2024-07-01"
    }
  },
  "id": 2
}

// Server Response with the result of the tool execution
{
  "jsonrpc": "2.0",
  "result": {
    "flights": [
      {
        "flight_number": "AA123",
        "departure_time": "2024-07-01T08:00:00",
        "arrival_time": "2024-07-01T11:00:00",
        "price": "$300"
      },
      {
        "flight_number": "UA456",
        "departure_time": "2024-07-01T09:00:00",
        "arrival_time": "2024-07-01T12:00:00",
        "price": "$350"
      }
    ]
  },
  "id": 2
}

```

<hr>

### **Shut Down**

The shutdown phase is initiated when either the client or server decides to terminate the connection. This can happen for various reasons, such as completion of tasks, user-initiated shutdown, or error conditions.

Generally the client initiates `Shutdown` by sending a `shutdown` request to the server, but no special `JSON RPC` shutdown message is defined in the protocol. Infact, `Transport Layer` is responsible for signaling shutdown through transport-specific mechanisms. For example, in the case of `STDIO` transport, the client can simply close the `stdin` stream to signal shutdown to the server. In the case of `Streaming HTTP` transport, the client can send an HTTP request to a specific shutdown endpoint or include a shutdown signal in the request body.


<hr>

## **Creating and Using an MCP Server**

To create and use, we need to download the `MCP SDK` for both client and the server. For example, if we want to create a local MCP server, we can use the `Python MCP SDK` to create a server that serves files from the local file system.

To use or call any tools or resources from the server, we need to create an `MCP Client` in the host application (e.g. `Claude Code`) and connect it to the server. Once connected, the host application can interact with the server by sending requests and receiving responses through the MCP client.

The tool call happens seamlessly from the host application, because both the client and server speak the same protocol and understand the same primitives. So when a client wants to call a tool on the server, it sends a request to the server using the MCP protocol. The server then processes the request, executes the tool, and sends back the response to the client. The client can then use this response to continue its operations or provide feedback to the user.

For example: 

**Client**

```python
from mcp import MCPClient
client = MCPClient()
client.connect("http://localhost:8000")
response = client.call_tool("read_file", {"path": "example.txt"})
print(response)
```

**Server**

```python
from mcp import MCPServer
server = MCPServer()
@server.tool("read_file")
def read_file(params):
    path = params["path"]
    with open(path, "r") as f:
        return f.read()
server.start(port=8000)
```

All the heavy lifting is done by the MCP SDK, which handles the communication, message formatting, and protocol adherence between the client and server. This allows developers to focus on implementing the specific logic of their tools and resources without worrying about the underlying communication details.

<hr>

To create a `MCP` server, we can use `FastMCP` - a Python framework for building MCP servers. Also, if we want to convert our `MCP` server into a `REST API`, we can use `FastAPI` - a modern, fast (high-performance) web framework for building APIs with Python. By combining `FastMCP` and `FastAPI`, we can create an MCP server that also exposes a RESTful API for external clients to interact with.

## *Creating Server**

**Note**

When we run the `MCP` server, by default the `Transport Layer` used is `STDIO`, which is designed for local communication between processes on the same machine. This means that the server will listen for incoming connections on the standard input/output streams, allowing it to communicate directly with clients that are also running on the same machine.

```python
from fastmcp import FastMCP

mcp = FastMCP("First MCP Server")

@mcp.tool
def greet(name: str) -> str:
    return f"Hello, {name}"

if __name__ == "__main__":
    mcp.run()
```

To run the server, we can use the command: 

```bash
python server.py
```

> This will run the file because the `__main__` block is executed when we run the script directly.

or with `FastMCP CLI`:

```bash
fastmcp run server.py:mcp
```

> Note if we use the `FastMCP CLI`, we need to specify the module and the MCP instance name (in this case, `mcp`) in the command. The `CLI` does not execute the `__main__` block, because it imports the server object i.e. `mcp` directly from the module, so it does not run the code inside the `if __name__ == "__main__":` block. Therefore, we need to ensure that the server is started by calling `mcp.run()` at the end of the script, outside of the `__main__` block, to ensure that it runs regardless of how the script is executed. 

To inspect the server and see the available tools, we can use the command: 

```bash
fastmcp dev inspector main.py:mcp
```

> The `MCP Inspector` is a powerful tool that allows developers to inspect and interact with their MCP servers in real-time. It provides a user-friendly interface to view the available tools, resources, and prompts exposed by the server, as well as the ability to test and debug interactions with the server. By using the `MCP Inspector`, developers can gain insights into how their server is functioning and identify any issues or areas for improvement.

<hr>

**But, But, But...**

> Is `MCP` obsolete? What about Context? Skills won?

## **MCP Vs SKILLS.md**

We know `MCP` server has a set of tools that any agentic harness can use. But, the thing is `MCP` is bloated with a lot of tools that are not relevant to the agentic harness, and most of the time only a few tools are relevant to the agentic harness. 

As the number of tools listed in the `MCP` server increases, it increases the context size for the `LLM` to process. A fraction of context size is used to define the tools, and their descriptions, and the rest of the context size is used to define the task. This leads to a situation where the `LLM` has to process a lot of irrelevant information, which can lead to a decrease in performance.

It’s not inherent in the protocol, but most MCP implementations today eagerly inject full tool schemas, which can bloat context.

**How SKILLS.md solves this problem?**

In production, there can be multiple `MCP` servers, each with a different set of tools. Passsig all the tool definitons from all the `MCP` servers to the `LLM` can lead to a bloated context, which can decrease the performance of the `LLM`.

Instead of sending the full schema for all the tools available in the `MCP` server, we can send only a thin description and some metadata coded in the `YAML` format, and a `PROGRESSIVE DISCLOSURE` mechanism can be used to fetch the full schema only when the `LLM` decides to use a specific tool. This way, we can keep the context size small and only fetch the relevant information when needed.

This has been well adopted by the `Agentic Harness` community, create a reference file called `SKILLS.md` that contains the description and metadata of the tools that are relevant to the agentic harness, and use a `PROGRESSIVE DISCLOSURE` mechanism to fetch the full schema only when needed. When the `LLM` decides to use a specific tool, it can fetch the full schema from the `MCP` server using the metadata provided in the `SKILLS.md` file.

> Very similar to lazy loading in programming, where you only load the necessary components when they are needed, rather than loading everything at once.

**What `MCP` got Right?**

But, `MCP` got something important right, and that's `AUTH`. The protocol has `auth`, especially `OAuth` baked into it, which means we don't need to support "Always Fresh Tokens", we don't need to provide single api token that can do anything - but we let it use user's own `OAuth` to a service (e.g. Google Calender, Jira, Notion) via their own login screens and security and automatically get a token with the users specific credentials for the specific service.